# Colab Internal Finetuning

Notebook này dùng cho 2 bước sau public pretraining:

- `clean_core`: finetune trên `2_face_dataset_clean_core_split`
- `full_ft`: finetune tiếp trên `2_face_dataset_split`

Giữ nguyên chiến lược hiện tại:
- AdaFace-only
- no KD
- no pruning


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Attendance_Workspace' / '3_edgeface_training'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
CLEAN_CORE_ROOT = DRIVE_ROOT / 'Attendance_Workspace' / '2_face_dataset_clean_core_split'
FULL_DATASET_ROOT = DRIVE_ROOT / 'Attendance_Workspace' / '2_face_dataset_split'

STAGE_PRESETS = {
    'clean_core': {
        'dataset_root': CLEAN_CORE_ROOT,
        'student_weights': CHECKPOINT_DIR / 'phase3_public_pretrain_best.pth',
        'epochs': 30,
        'batch_size': 64,
        'num_workers': 2,
        'learning_rate': 5e-5,
        'output_prefix': 'phase3_public_to_clean_core',
        'max_train_batches_per_epoch': None,
        'max_val_batches': None,
    },
    'full_ft': {
        'dataset_root': FULL_DATASET_ROOT,
        'student_weights': CHECKPOINT_DIR / 'phase3_public_to_clean_core_best.pth',
        'epochs': 10,
        'batch_size': 64,
        'num_workers': 2,
        'learning_rate': 5e-5,
        'output_prefix': 'phase3_cleancore_to_full_ft',
        'max_train_batches_per_epoch': None,
        'max_val_batches': None,
    },
}

ACTIVE_STAGE = 'clean_core'
WIDTH_PRESET = 'widened'
RANK_RATIO = 0.7

cfg = STAGE_PRESETS[ACTIVE_STAGE]
print('ACTIVE_STAGE =', ACTIVE_STAGE)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASET_ROOT =', cfg['dataset_root'])
print('STUDENT_WEIGHTS =', cfg['student_weights'])
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)


In [ ]:
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert CHECKPOINT_DIR.exists(), f'Checkpoint dir not found: {CHECKPOINT_DIR}'
assert cfg['dataset_root'].exists(), f'Dataset root not found: {cfg["dataset_root"]}'
assert cfg['student_weights'].exists(), f'Student weights not found: {cfg["student_weights"]}'
assert (cfg['dataset_root'] / 'train').exists(), f'Missing train split: {cfg["dataset_root"] / "train"}'
assert (cfg['dataset_root'] / 'val').exists(), f'Missing val split: {cfg["dataset_root"] / "val"}'


In [ ]:
%cd {PROJECT_ROOT}
!pip install -q -r requirements.txt


In [ ]:
import shlex
import subprocess

cmd = [
    'python',
    'scripts/train_phase3.py',
    '--dataset-root', str(cfg['dataset_root']),
    '--checkpoints-dir', str(CHECKPOINT_DIR),
    '--student-weights', str(cfg['student_weights']),
    '--epochs', str(cfg['epochs']),
    '--batch-size', str(cfg['batch_size']),
    '--num-workers', str(cfg['num_workers']),
    '--learning-rate', str(cfg['learning_rate']),
    '--width-preset', WIDTH_PRESET,
    '--rank-ratio', str(RANK_RATIO),
    '--kd-alpha', '0',
    '--skip-teacher-bootstrap',
    '--output-prefix', cfg['output_prefix'],
]
if cfg['max_train_batches_per_epoch'] is not None:
    cmd += ['--max-train-batches-per-epoch', str(cfg['max_train_batches_per_epoch'])]
if cfg['max_val_batches'] is not None:
    cmd += ['--max-val-batches', str(cfg['max_val_batches'])]

print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)


## Recommended Use

- Chạy `clean_core` trước
- Chỉ chạy `full_ft` sau khi đã có `phase3_public_to_clean_core_best.pth`
- Không bật KD trong notebook này
